# ================================================================
# BOOTCAMP: Fundamentos de Ingeniería de Datos
# Semana 1: Práctica de JOINs en SQL
# ================================================================

**Instructor:** Luciano Argolo  
**Web:** lucianoargolo.com

---

## 🎯 Objetivos

1. Entender **qué es un JOIN** y para qué sirve
2. Dominar los diferentes tipos: **INNER, LEFT, RIGHT, FULL OUTER, SELF**
3. Practicar con datos de ejemplo fáciles de entender
4. Saber cuándo usar cada tipo de JOIN

---

## 📊 Escenario de Práctica

Vamos a trabajar con un **sistema de e-commerce** simplificado:
- **Clientes** → quiénes compran
- **Órdenes** → qué compraron
- **Productos** → catálogo disponible
- **Empleados** → con jerarquía (para SELF JOIN)


# ================================================================
# MÓDULO 1: DATOS DE PRÁCTICA
# ================================================================

Creamos las tablas de práctica en `bootcamp.practice`. 


In [0]:
%sql
-- ============================================================
-- SETUP: Crear tablas de práctica para JOINs
-- Ejecutar UNA VEZ antes de la clase
-- ============================================================


CREATE CATALOG IF NOT EXISTS BOOTCAMP;


CREATE SCHEMA IF NOT EXISTS bootcamp.practice;

-- CLIENTES
CREATE OR REPLACE TABLE bootcamp.practice.clientes (
    cliente_id INT,
    nombre STRING,
    email STRING,
    ciudad STRING
);

INSERT INTO bootcamp.practice.clientes VALUES
 (1, 'Ana',    'ana@email.com',    'Buenos Aires'),
  (2, 'Carlos', 'carlos@email.com', 'Córdoba'),
  (3, 'María',  'maria@email.com',  'Buenos Aires'),
  (4, 'Diego',  'diego@email.com',  'Rosario'),
  (5, 'Laura',  'laura@email.com',  'Mendoza'),
  (6, 'Pedro',  'pedro@email.com',  'Buenos Aires');  -- Sin órdenes (para LEFT JOIN)

-- PRODUCTOS
CREATE OR REPLACE TABLE bootcamp.practice.productos (
    producto_id INT,
    nombre STRING,
    categoria STRING,
    precio DOUBLE
);

INSERT INTO bootcamp.practice.productos VALUES
(101, 'Laptop Pro', 'Electrónica', 1200.00),
(102, 'Mouse Inalámbrico', 'Electrónica', 35.00),
(103, 'Teclado Mecánico', 'Electrónica', 89.00),
(104, 'Monitor 27"', 'Electrónica', 450.00),
(105, 'Auriculares BT', 'Audio', 75.00),
(106, 'Webcam HD', 'Accesorios', 55.00),
(107, 'Hub USB-C', 'Accesorios', 42.00),
(108, 'Silla Gamer', 'Muebles', 380.00);  -- Nunca vendida (para LEFT JOIN)

-- ORDENES
CREATE OR REPLACE TABLE bootcamp.practice.ordenes (
    orden_id INT,
    cliente_id INT,
    producto_id INT,
    cantidad INT,
    fecha DATE
);

INSERT INTO bootcamp.practice.ordenes VALUES
(1001, 1, 101, 1, '2024-01-15'),
(1002, 1, 102, 2, '2024-01-15'),
(1003, 2, 103, 1, '2024-02-01'),
(1004, 3, 101, 1, '2024-02-10'),
(1005, 3, 105, 3, '2024-02-10'),
(1006, 4, 104, 1, '2024-03-05'),
(1007, 5, 106, 2, '2024-03-12'),
(1008, 5, 107, 1, '2024-03-12'),
(1009, 2, 102, 1, '2024-04-01'),
(1010, 99, 101, 1, '2024-04-15'),  -- cliente_id=99 NO existe (para RIGHT/FULL JOIN)
(1011, 1, 999, 2, '2024-05-01');   -- producto_id=999 NO existe (para integridad)

-- EMPLEADOS (jerarquía para SELF JOIN)
CREATE OR REPLACE TABLE bootcamp.practice.empleados (
    empleado_id INT,
    nombre STRING,
    cargo STRING,
    jefe_id INT
);

INSERT INTO bootcamp.practice.empleados VALUES
(1, 'Roberto', 'CEO', NULL),           -- Sin jefe (nivel 1)
(2, 'Silvia', 'Directora de Datos', 1),
(3, 'Martín', 'Director Comercial', 1),
(4, 'Lucía', 'Data Engineer Sr', 2),
(5, 'Pablo', 'Data Engineer Jr', 4),
(6, 'Camila', 'Data Analyst', 2),
(7, 'Tomás', 'Vendedor Sr', 3),
(8, 'Valentina', 'Vendedora Jr', 3);

In [0]:
%sql
-- ============================================================
-- Verificar tabla: bootcamp.practice.clientes
-- ============================================================
-- Estas tablas están pre-creadas en bootcamp.practice
-- Si no existen, ejecutar el notebook de setup

SELECT * FROM bootcamp.practice.clientes;

In [0]:
%sql
-- ============================================================
-- Verificar tabla: bootcamp.practice.productos
-- ============================================================
-- Estas tablas están pre-creadas en bootcamp.practice
-- Si no existen, ejecutar el notebook de setup

SELECT * FROM bootcamp.practice.productos;

In [0]:
%sql
-- ============================================================
-- Verificar tabla: bootcamp.practice.ordenes
-- ============================================================
-- Estas tablas están pre-creadas en bootcamp.practice
-- Si no existen, ejecutar el notebook de setup

SELECT * FROM bootcamp.practice.ordenes;

In [0]:
%sql
-- ============================================================
-- Verificar tabla: bootcamp.practice.empleados
-- ============================================================
-- Estas tablas están pre-creadas en bootcamp.practice
-- Si no existen, ejecutar el notebook de setup

SELECT * FROM bootcamp.practice.empleados;

# ================================================================
# MÓDULO 2: INNER JOIN
# ================================================================

## ¿Qué es INNER JOIN?

Devuelve **SOLO las filas que tienen coincidencia en AMBAS tablas**.

```
    TABLA A          TABLA B
   ┌───────┐        ┌───────┐
   │       │        │       │
   │   ████████████████     │
   │       │        │       │
   └───────┘        └───────┘
           ↑
    Solo la intersección
```

**Caso de uso:** Cuando SOLO querés datos que existen en ambos lados.

**Ejemplo:** "Mostrar clientes CON sus órdenes" (ignora clientes sin órdenes)


In [0]:
%sql
-- ============================================================
-- Ejercicio 2.1: Clientes con sus órdenes
-- ============================================================
-- RESULTADO: Solo clientes que tienen órdenes
--            Pedro (cliente_id=6) NO aparece porque no tiene órdenes
--            La orden 1010 (cliente_id=99) NO aparece porque el cliente no existe

SELECT 
    c.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    o.orden_id,
    o.producto_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.clientes c
INNER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY c.cliente_id, o.orden_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 2.2: Órdenes con detalle de producto
-- ============================================================
-- Muestra nombre del producto y calcula el total
-- La orden 1011 (producto_id=999) NO aparece porque el producto no existe

SELECT 
    o.orden_id,
    o.fecha,
    p.nombre as producto,
    p.categoria,
    o.cantidad,
    p.precio as precio_unitario,
    o.cantidad * p.precio as total
FROM bootcamp.practice.ordenes o
INNER JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
ORDER BY o.orden_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 2.3: Reporte completo (Clientes + Órdenes + Productos)
-- ============================================================
-- Reporte completo: quién compró qué y cuánto gastó

SELECT 
    c.nombre as cliente,
    c.ciudad,
    o.orden_id,
    o.fecha,
    p.nombre as producto,
    p.categoria,
    o.cantidad,
    p.precio as precio_unitario,
    o.cantidad * p.precio as total
FROM bootcamp.practice.clientes c
INNER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
INNER JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
ORDER BY c.nombre, o.fecha;


# ================================================================
# MÓDULO 3: LEFT JOIN (LEFT OUTER JOIN)
# ================================================================

## ¿Qué es LEFT JOIN?

Devuelve **TODAS las filas de la tabla IZQUIERDA** + las coincidencias de la derecha.
Si no hay coincidencia, muestra NULL en las columnas de la derecha.

```
    TABLA A          TABLA B
   ┌───────┐        ┌───────┐
   │███████████████████     │
   │███████│        │       │
   │███████│        │       │
   └───────┘        └───────┘
       ↑
   TODO de A + coincidencias de B
```

**Caso de uso:** Cuando querés TODOS los registros de una tabla, tengan o no coincidencia.

**Ejemplo:** "Mostrar TODOS los clientes, hayan comprado o no"


In [0]:
%sql
-- ============================================================
-- Ejercicio 3.1: Todos los clientes con sus órdenes
-- ============================================================
-- RESULTADO: Todos los clientes aparecen
--            Pedro (cliente_id=6) aparece con NULL en las columnas de órdenes

SELECT 
    c.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    o.orden_id,
    o.producto_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY c.cliente_id, o.orden_id;


In [0]:
%sql
-- ============================================================
-- LEFT OUTER JOIN: Solo clientes que tienen órdenes
-- ============================================================
-- DIFERENCIA: LEFT OUTER JOIN trae SOLO los de la izquierda con match
-- Se logra con LEFT JOIN + WHERE tabla_derecha.id IS NOT NULL

SELECT 
    c.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    o.orden_id,
    o.producto_id
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
WHERE o.orden_id IS NOT NULL  -- Solo los que tienen órdenes
ORDER BY c.cliente_id, o.orden_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 3.2: Clientes que nunca compraron
-- ============================================================
-- Truco: Filtrar donde la columna de la tabla derecha es NULL

SELECT 
    c.cliente_id,
    c.nombre as cliente,    
    c.email,
    c.ciudad
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
WHERE o.orden_id IS NULL;  -- Solo los que NO tienen órdenes


In [0]:
%sql
-- ============================================================
-- Ejercicio 3.3: Todos los productos vendidos o no
-- ============================================================
-- La Silla Gamer (producto_id=108) aparece con NULL porque nunca se vendió

SELECT 
    p.producto_id,
    p.nombre as producto,
    p.categoria,
    p.precio,
    o.orden_id,
    o.cantidad,
    COALESCE(o.cantidad * p.precio, 0) as total_vendido
FROM bootcamp.practice.productos p
LEFT JOIN bootcamp.practice.ordenes o ON p.producto_id = o.producto_id
ORDER BY p.producto_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 3.4: Productos nunca vendidos
-- ============================================================

SELECT 
    p.producto_id,
    p.nombre as producto,
    p.categoria,
    p.precio
FROM bootcamp.practice.productos p
LEFT JOIN bootcamp.practice.ordenes o ON p.producto_id = o.producto_id
WHERE o.orden_id IS NULL;  -- Solo productos sin ventas


# ================================================================
# MÓDULO 4: RIGHT JOIN vs RIGHT OUTER JOIN
# ================================================================

## ¿Qué es RIGHT JOIN?

Devuelve **TODAS las filas de la tabla DERECHA** + las coincidencias de la izquierda.
Si no hay coincidencia, muestra NULL en las columnas de la izquierda.

```
    TABLA A          TABLA B
   ┌───────┐        ┌───────┐
   │       ████████████████│
   │       │        │██████│
   │       │        │██████│
   └───────┘        └───────┘
                        ↑
       Coincidencias de A + TODO de B
```

**Pro tip:** Un `RIGHT JOIN` siempre se puede reescribir como `LEFT JOIN` invirtiendo el orden de las tablas. Por eso en la práctica se usa más `LEFT JOIN`.


**Caso de uso:** Cuando querés SOLO los registros de la derecha que tienen relación.

**Ejemplo:** "Mostrar SOLO las órdenes que tienen cliente válido"


In [0]:
%sql
-- ============================================================
-- Ejercicio 4.1: Todas las órdenes con cliente
-- ============================================================
-- RESULTADO: Todas las órdenes aparecen
--            La orden 1010 (cliente_id=99) aparece con NULL en datos de cliente

SELECT 
    o.orden_id,
    o.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    o.producto_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.clientes c
RIGHT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY o.orden_id;


In [0]:
%sql
-- ============================================================
-- RIGHT OUTER JOIN: Solo órdenes que tienen cliente válido
-- ============================================================
-- DIFERENCIA: RIGHT OUTER JOIN trae SOLO los de la derecha con match
-- Se logra con RIGHT JOIN + WHERE tabla_izquierda.id IS NOT NULL

SELECT 
    o.orden_id,
    o.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    o.producto_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.clientes c
RIGHT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
WHERE c.cliente_id IS NOT NULL  -- Solo órdenes con cliente válido
ORDER BY o.orden_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 4.2: Órdenes huérfanas (cliente no existe)
-- ============================================================
-- Útil para detectar problemas de integridad referencial

SELECT 
    o.orden_id,
    o.cliente_id as cliente_id_inexistente,
    o.producto_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.clientes c
RIGHT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
WHERE c.cliente_id IS NULL;  -- El cliente NO existe


In [0]:
%sql
-- ============================================================
-- Ejercicio 4.3: Equivalencia LEFT vs RIGHT (versión RIGHT)
-- ============================================================
-- Estas dos queries dan el MISMO resultado:

-- Versión con RIGHT JOIN
SELECT o.orden_id, o.cliente_id, c.nombre
FROM bootcamp.practice.clientes c
RIGHT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY o.orden_id;


In [0]:
%sql
-- Ejercicio 4.3 (cont.): Versión equivalente con LEFT JOIN
SELECT o.orden_id, o.cliente_id, c.nombre
FROM bootcamp.practice.ordenes o
LEFT JOIN bootcamp.practice.clientes c ON o.cliente_id = c.cliente_id
ORDER BY o.orden_id;


# ================================================================
# MÓDULO 5: FULL OUTER JOIN
# ================================================================

## ¿Qué es FULL OUTER JOIN?

Devuelve **TODAS las filas de AMBAS tablas**, tengan o no coincidencia.
Donde no hay coincidencia, muestra NULL.

```
    TABLA A          TABLA B
   ┌───────┐        ┌───────┐
   │███████████████████████│
   │███████│        │██████│
   │███████│        │██████│
   └───────┘        └───────┘
       ↑                ↑
   TODO de A    +   TODO de B
```

**Caso de uso:** Cuando querés ver TODO de ambos lados, identificando qué no tiene match.

**Ejemplo:** "Ver todos los clientes Y todas las órdenes, identificando huérfanos de ambos lados"


In [0]:
%sql
-- ============================================================
-- Ejercicio 5.1: Todos los clientes y todas las órdenes
-- ============================================================
-- RESULTADO: 
--   - Pedro (cliente_id=6) aparece con NULL en órdenes (nunca compró)
--   - Orden 1010 (cliente_id=99) aparece con NULL en cliente (no existe)

SELECT 
    c.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    o.orden_id,
    o.producto_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.clientes c
FULL OUTER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY COALESCE(c.cliente_id, 999), o.orden_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 5.2: Detectar problemas de datos
-- ============================================================
-- Clasificar cada fila según si tiene match o no

SELECT 
    COALESCE(c.cliente_id, o.cliente_id) as cliente_id,
    c.nombre as cliente,
    o.orden_id,
    CASE 
        WHEN c.cliente_id IS NOT NULL AND o.orden_id IS NOT NULL THEN '✅ Match correcto'
        WHEN c.cliente_id IS NOT NULL AND o.orden_id IS NULL THEN '⚠️ Cliente sin órdenes'
        WHEN c.cliente_id IS NULL AND o.orden_id IS NOT NULL THEN '❌ Orden huérfana'
    END as estado
FROM bootcamp.practice.clientes c
FULL OUTER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
ORDER BY 
    CASE 
        WHEN c.cliente_id IS NULL THEN 3  -- Huérfanas al final
        WHEN o.orden_id IS NULL THEN 2    -- Sin órdenes después
        ELSE 1                             -- Matches primero
    END,
    c.cliente_id;


In [0]:
%sql
-- ============================================================
-- FULL OUTER JOIN: Productos vs Órdenes
-- ============================================================
-- Ver qué productos se vendieron y cuáles órdenes tienen producto inválido

SELECT 
    COALESCE(p.producto_id, o.producto_id) as producto_id,
    p.nombre as producto,
    p.categoria,
    o.orden_id,
    o.cantidad,
    CASE 
        WHEN p.producto_id IS NOT NULL AND o.orden_id IS NOT NULL THEN 'Vendido'
        WHEN p.producto_id IS NOT NULL AND o.orden_id IS NULL THEN 'Sin ventas'
        WHEN p.producto_id IS NULL AND o.orden_id IS NOT NULL THEN 'Producto inexistente'
    END as estado
FROM bootcamp.practice.productos p
FULL OUTER JOIN bootcamp.practice.ordenes o ON p.producto_id = o.producto_id
ORDER BY estado, producto_id;


# ================================================================
# MÓDULO 6: SELF JOIN
# ================================================================

## ¿Qué es SELF JOIN?

Es cuando **una tabla se une consigo misma**. Se usa cuando una fila referencia a otra fila de la **misma tabla**.

**Casos de uso comunes:**
- 📊 Jerarquías (empleados con jefes)
- 🔗 Relaciones padre-hijo en la misma tabla
- 📈 Comparar filas de la misma tabla entre sí

**Sintaxis:** Se usa un alias diferente para cada "copia" de la tabla.

```sql
SELECT e.nombre as empleado, j.nombre as jefe
FROM empleados e
LEFT JOIN empleados j ON e.jefe_id = j.empleado_id
```


In [0]:
%sql
-- ============================================================
-- Recordar: Nuestra tabla de empleados con jerarquía
-- ============================================================

SELECT * FROM bootcamp.practice.empleados ORDER BY empleado_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 6.1: Jerarquía de empleados
-- ============================================================
-- Muestra cada empleado con el nombre de su jefe
-- Usamos LEFT JOIN porque el CEO no tiene jefe (jefe_id = NULL)

SELECT 
    e.empleado_id,
    e.nombre as empleado,
    e.cargo,
    e.jefe_id,
    j.nombre as jefe
FROM bootcamp.practice.empleados e
LEFT JOIN bootcamp.practice.empleados j ON e.jefe_id = j.empleado_id
ORDER BY e.empleado_id;


In [0]:
%sql
-- ============================================================
-- Ejercicio 6.2: Empleados sin jefe (nivel más alto)
-- ============================================================
-- ¿Qué empleado no tiene jefe? → El CEO

SELECT 
    e.empleado_id,
    e.nombre as empleado,
    e.cargo
FROM bootcamp.practice.empleados e
LEFT JOIN bootcamp.practice.empleados j ON e.jefe_id = j.empleado_id
WHERE e.jefe_id IS NULL;

In [0]:
%sql
-- ============================================================
-- Ejercicio 6.3: Subordinados directos
-- ============================================================
-- Muestra quién reporta directamente a cada jefe

SELECT 
    j.nombre as jefe,
    j.cargo as cargo_jefe,
    e.nombre as subordinado,
    e.cargo as cargo_subordinado
FROM bootcamp.practice.empleados e
INNER JOIN bootcamp.practice.empleados j ON e.jefe_id = j.empleado_id
ORDER BY j.nombre, e.nombre;

In [0]:
%sql
-- ============================================================
-- Extra - SELF JOIN: Mostrar empleados con el jefe de su jefe (abuelo)
-- ============================================================
-- Encadenamos múltiples SELF JOINs

SELECT 
    e.nombre as empleado,
    e.cargo,
    j1.nombre as jefe_directo,
    j2.nombre as jefe_del_jefe
FROM bootcamp.practice.empleados e
LEFT JOIN bootcamp.practice.empleados j1 ON e.jefe_id = j1.empleado_id         -- Jefe directo
LEFT JOIN bootcamp.practice.empleados j2 ON j1.jefe_id = j2.empleado_id        -- Jefe del jefe
ORDER BY e.empleado_id;


In [0]:
%sql
-- ============================================================
-- Extra - SELF JOIN: Contar cuántos reportes directos tiene cada jefe
-- ============================================================

SELECT 
    j.empleado_id,
    j.nombre as jefe,
    j.cargo,
    COUNT(e.empleado_id) as cantidad_reportes
FROM bootcamp.practice.empleados j
LEFT JOIN bootcamp.practice.empleados e ON j.empleado_id = e.jefe_id  -- e reporta a j
GROUP BY j.empleado_id, j.nombre, j.cargo
ORDER BY cantidad_reportes DESC;


In [0]:
%sql
-- ============================================================
-- Extra - SELF JOIN: Encontrar compañeros de equipo (mismo jefe)
-- ============================================================

SELECT 
    e1.nombre as empleado,
    e2.nombre as companero,
    j.nombre as jefe_en_comun
FROM bootcamp.practice.empleados e1
INNER JOIN bootcamp.practice.empleados e2 ON e1.jefe_id = e2.jefe_id   -- Mismo jefe
    AND e1.empleado_id < e2.empleado_id              -- Evitar duplicados y auto-match
LEFT JOIN bootcamp.practice.empleados j ON e1.jefe_id = j.empleado_id
ORDER BY jefe_en_comun, empleado;


In [0]:
%sql
-- ============================================================
-- Extra - SELF JOIN: Organigrama completo (path jerárquico)
-- ============================================================
-- Mostrar la cadena completa de mando para cada empleado

SELECT 
    e.nombre as empleado,
    e.cargo,
    CONCAT_WS(' → ', 
        j3.nombre,  -- Nivel 3 arriba
        j2.nombre,  -- Nivel 2 arriba
        j1.nombre,  -- Jefe directo
        e.nombre    -- El empleado
    ) as cadena_de_mando
FROM bootcamp.practice.empleados e
LEFT JOIN bootcamp.practice.empleados j1 ON e.jefe_id = j1.empleado_id
LEFT JOIN bootcamp.practice.empleados j2 ON j1.jefe_id = j2.empleado_id
LEFT JOIN bootcamp.practice.empleados j3 ON j2.jefe_id = j3.empleado_id
ORDER BY e.empleado_id;


# ================================================================
# MÓDULO 7: JOINs MÚLTIPLES
# ================================================================

Combinar múltiples JOINs en una sola consulta es una habilidad esencial en ingeniería de datos.
Permite construir reportes ricos a partir de tablas relacionadas.

**Buenas prácticas:**
- Usá siempre alias de tabla cortos y claros
- Aplicá `COALESCE()` para convertir NULLs en ceros cuando necesites métricas de registros sin coincidencia
- Ordená el resultado para facilitar la lectura

In [0]:
%sql
-- ============================================================
-- Ejercicio 7.1: Reporte completo con 3 tablas
-- ============================================================
-- Nombre del cliente, ciudad, orden_id, fecha, nombre del producto,
-- categoría, cantidad, precio unitario, total. Ordena por cliente y fecha.

SELECT 
    c.nombre as cliente,
    c.ciudad,
    o.orden_id,
    o.fecha,
    p.nombre as producto,
    p.categoria,
    o.cantidad,
    p.precio as precio_unitario,
    o.cantidad * p.precio as total
FROM bootcamp.practice.clientes c
INNER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
INNER JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
ORDER BY c.nombre, o.fecha;

In [0]:
%sql
-- ============================================================
-- Ejercicio 7.2: Resumen por cliente
-- ============================================================
-- Para cada cliente: nombre, ciudad, cantidad total de órdenes, monto total gastado
-- LEFT JOIN para incluir clientes sin órdenes (Pedro aparece con 0)

SELECT 
    c.nombre as cliente,
    c.ciudad,
    COUNT(o.orden_id) as cantidad_ordenes,
    COALESCE(SUM(o.cantidad * p.precio), 0) as monto_total_gastado
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
LEFT JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
GROUP BY c.cliente_id, c.nombre, c.ciudad
ORDER BY monto_total_gastado DESC;

In [0]:
%sql
-- ============================================================
-- Ejercicio 7.3: Resumen por producto
-- ============================================================
-- Para cada producto: nombre, categoría, cantidad total vendida, ingresos totales
-- Incluye productos que nunca se vendieron (con 0)

SELECT 
    p.nombre as producto,
    p.categoria,
    COALESCE(SUM(o.cantidad), 0) as cantidad_total_vendida,
    COALESCE(SUM(o.cantidad * p.precio), 0) as ingresos_totales
FROM bootcamp.practice.productos p
LEFT JOIN bootcamp.practice.ordenes o ON p.producto_id = o.producto_id
GROUP BY p.producto_id, p.nombre, p.categoria
ORDER BY ingresos_totales DESC;

# ================================================================
# MÓDULO 8: RESUMEN Y CUÁNDO USAR CADA JOIN
# ================================================================

## 📋 Tabla resumen

| JOIN | Qué devuelve | Uso típico |
|------|--------------|------------|
| **INNER JOIN** | Solo coincidencias | Reportes donde necesitás datos completos |
| **LEFT JOIN** | Todo de izquierda + matches | Incluir registros sin relación (clientes sin órdenes) |
| **LEFT OUTER JOIN** | Solo de izquierda CON match | Solo clientes que tienen órdenes (WHERE derecha IS NOT NULL) |
| **RIGHT JOIN** | Todo de derecha + matches | Poco usado, se reemplaza con LEFT invirtiendo |
| **RIGHT OUTER JOIN** | Solo de derecha CON match | Solo órdenes con cliente válido (WHERE izquierda IS NOT NULL) |
| **FULL OUTER JOIN** | Todo de ambos lados | Auditoría, detectar huérfanos |
| **SELF JOIN** | Tabla consigo misma | Jerarquías, comparar filas |

---

## 🎯 Tips prácticos

1. **Empezá con INNER JOIN** si solo necesitás matches
2. **Usá LEFT JOIN** cuando querés "todo de esta tabla, y si hay datos relacionados, mejor"
3. **LEFT OUTER JOIN** = LEFT JOIN + WHERE derecha IS NOT NULL (solo los que tienen match)
4. **RIGHT OUTER JOIN** = RIGHT JOIN + WHERE izquierda IS NOT NULL (solo los que tienen match)
5. **FULL OUTER JOIN** es útil para detectar problemas de integridad
6. **SELF JOIN** siempre necesita alias diferentes para la misma tabla
7. **RIGHT JOIN** casi siempre se puede reescribir como LEFT JOIN

---

## ⚠️ Errores comunes

1. **Olvidar la condición ON** → Producto cartesiano (miles de filas)
2. **JOIN sin índices** → Queries muy lentas en tablas grandes
3. **Confundir INNER con LEFT** → Perder datos importantes
4. **No usar alias** → Queries confusas y ambiguas


# ================================================================
# EJERCICIOS PARA PRACTICAR
# ================================================================

Intentá resolver estos ejercicios usando las tablas que creamos:

### Nivel 1 🟢
1. Mostrar todas las órdenes con el nombre del producto (INNER JOIN)
2. Mostrar todos los clientes con la cantidad de órdenes que hizo cada uno
3. Listar productos que nunca fueron vendidos

### Nivel 2 🟡
4. Crear un reporte con: cliente, producto, cantidad, total (unir 3 tablas)
5. Encontrar órdenes que tienen producto inválido (producto no existe)
6. Calcular el gasto total de cada cliente

### Nivel 3 🔴
7. Para cada empleado, mostrar su nivel en la jerarquía (CEO=1, Directora=2, etc.)
8. Encontrar empleados que tienen al menos 2 reportes directos
9. Crear un reporte de integridad que muestre:
   - Clientes sin órdenes
   - Órdenes con cliente inválido
   - Productos sin ventas

---

**¡Intentá resolverlos antes de ver las soluciones abajo!**


In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 1: Órdenes con nombre de producto
-- ============================================================

SELECT 
    o.orden_id,
    o.fecha,
    p.nombre as producto,
    o.cantidad,
    p.precio,
    o.cantidad * p.precio as total
FROM bootcamp.practice.ordenes o
INNER JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
ORDER BY o.orden_id;


In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 2: Clientes con cantidad de órdenes
-- ============================================================

SELECT 
    c.cliente_id,
    c.nombre as cliente,
    COUNT(o.orden_id) as cantidad_ordenes
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
GROUP BY c.cliente_id, c.nombre
ORDER BY cantidad_ordenes DESC;


In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 6: Gasto total por cliente
-- ============================================================

SELECT 
    c.cliente_id,
    c.nombre as cliente,
    c.ciudad,
    COUNT(o.orden_id) as cantidad_ordenes,
    COALESCE(SUM(o.cantidad * p.precio), 0) as gasto_total
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
LEFT JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
GROUP BY c.cliente_id, c.nombre, c.ciudad
ORDER BY gasto_total DESC;


In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 3: Productos que nunca fueron vendidos
-- ============================================================

SELECT 
    p.producto_id,
    p.nombre as producto,
    p.categoria,
    p.precio
FROM bootcamp.practice.productos p
LEFT JOIN bootcamp.practice.ordenes o ON p.producto_id = o.producto_id
WHERE o.orden_id IS NULL;

In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 4: Reporte completo (3 tablas)
-- ============================================================

SELECT 
    c.nombre as cliente,
    c.ciudad,
    p.nombre as producto,
    p.categoria,
    o.cantidad,
    p.precio as precio_unitario,
    o.cantidad * p.precio as total
FROM bootcamp.practice.clientes c
INNER JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
INNER JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
ORDER BY c.nombre, o.fecha;

In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 5: Órdenes con producto inválido
-- ============================================================

SELECT 
    o.orden_id,
    o.producto_id as producto_id_inexistente,
    o.cliente_id,
    o.cantidad,
    o.fecha
FROM bootcamp.practice.ordenes o
LEFT JOIN bootcamp.practice.productos p ON o.producto_id = p.producto_id
WHERE p.producto_id IS NULL;

In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 7: Nivel jerárquico de cada empleado
-- ============================================================

SELECT 
    e.empleado_id,
    e.nombre,
    e.cargo,
    CASE 
        WHEN e.jefe_id IS NULL THEN 1
        WHEN j1.jefe_id IS NULL THEN 2
        WHEN j2.jefe_id IS NULL THEN 3
        ELSE 4
    END as nivel
FROM bootcamp.practice.empleados e
LEFT JOIN bootcamp.practice.empleados j1 ON e.jefe_id = j1.empleado_id
LEFT JOIN bootcamp.practice.empleados j2 ON j1.jefe_id = j2.empleado_id
ORDER BY nivel, e.empleado_id;

In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 8: Empleados con al menos 2 reportes directos
-- ============================================================

SELECT 
    j.empleado_id,
    j.nombre as jefe,
    j.cargo,
    COUNT(e.empleado_id) as reportes_directos
FROM bootcamp.practice.empleados j
INNER JOIN bootcamp.practice.empleados e ON j.empleado_id = e.jefe_id
GROUP BY j.empleado_id, j.nombre, j.cargo
HAVING COUNT(e.empleado_id) >= 2
ORDER BY reportes_directos DESC;

In [0]:
%sql
-- ============================================================
-- SOLUCIÓN Ejercicio 9: Reporte de integridad completo
-- ============================================================

SELECT 'Clientes sin órdenes' as tipo_problema, 
    c.cliente_id as id, 
    c.nombre as detalle
FROM bootcamp.practice.clientes c
LEFT JOIN bootcamp.practice.ordenes o ON c.cliente_id = o.cliente_id
WHERE o.orden_id IS NULL

UNION ALL

SELECT 'Órdenes con cliente inválido' as tipo_problema, 
    o.orden_id as id, 
    CONCAT('cliente_id: ', o.cliente_id) as detalle
FROM bootcamp.practice.ordenes o
LEFT JOIN bootcamp.practice.clientes c ON o.cliente_id = c.cliente_id
WHERE c.cliente_id IS NULL

UNION ALL

SELECT 'Productos sin ventas' as tipo_problema, 
    p.producto_id as id, 
    p.nombre as detalle
FROM bootcamp.practice.productos p
LEFT JOIN bootcamp.practice.ordenes o ON p.producto_id = o.producto_id
WHERE o.orden_id IS NULL

ORDER BY tipo_problema, id;

---

## ✅ Checklist de finalización

Antes de pasar al siguiente tema, asegurate de entender:

- [ ] Qué hace cada tipo de JOIN
- [ ] La diferencia entre INNER y LEFT JOIN
- [ ] La diferencia entre LEFT JOIN y LEFT OUTER JOIN (LEFT OUTER = solo con match)
- [ ] La diferencia entre RIGHT JOIN y RIGHT OUTER JOIN (RIGHT OUTER = solo con match)
- [ ] Cómo detectar registros sin match usando WHERE ... IS NULL
- [ ] Cómo funciona SELF JOIN para jerarquías
- [ ] Cuándo usar cada tipo de JOIN

---

*Bootcamp: Fundamentos de Ingeniería de Datos | Instructor: Luciano Argolo | [lucianoargolo.com](https://lucianoargolo.com)*
